# Data Mining (AIT3003) - Problem Set 2: Data Preparation

This notebook contains the complete, reproducible solutions, code implementations, mathematical calculations, and visualizations for **Problem Set 2: Data Preparation**.

Reference: *Data Mining: Concepts and Techniques (3rd Edition)* by Jiawei Han, Micheline Kamber, and Jian Pei (Morgan Kaufmann).


## Problem P1: Handling Missing Values in Real-World Data

In real-world data mining scenarios, missing values commonly occur due to equipment faults, human omission, schema mismatches during data integration, or conditional non-applicability.

### Key Methods for Handling Missing Values (Han & Kamber, Section 3.2.1):
1. **Ignore the tuple (Listwise Deletion):** Discard records containing missing attributes. Highly effective when the *class label* is missing in supervised classification. However, results in severe data loss if missingness is prevalent across multiple attributes.
2. **Fill in missing values manually:** Domain experts manually inspect and supply missing values. High precision, but expensive and completely unfeasible for big data.
3. **Use a global constant (e.g., 'Unknown', -999, 'N/A'):** Simple and fast, but data mining algorithms may treat the sentinel constant as an interesting actual attribute value or cluster.
4. **Use attribute central tendency measures (Mean or Median):** 
   - **Mean:** Preferred for symmetric, bell-shaped (normal) distributions.
   - **Median:** Preferred for skewed distributions or datasets containing severe outliers.
5. **Use class-conditional attribute mean or median:** Impute using the central tendency computed only across samples belonging to the same class/cluster.
6. **Use the most probable value (Predictive / Model-based Imputation):** Impute via regression models, decision trees (e.g., missForest), or $k$-Nearest Neighbors ($k$-NN) based on inter-attribute correlations.
7. **Multiple Imputation (e.g., MICE):** Generate multiple completed datasets to capture imputation uncertainty.


## Problem P2: Data Smoothing & Outlier Detection for Age Attribute

Exercise 1 gave the following data (in increasing order) for the attribute age:
$$\{13, 15, 16, 16, 19, 20, 20, 21, 22, 22, 25, 25, 25, 25, 30, 33, 33, 35, 35, 35, 35, 36, 40, 45, 46, 52, 70\}$$

### (a) Smoothing by Bin Means (Bin Depth = 3)
Equal-frequency binning with depth $d = 3$ creates $27 / 3 = 9$ bins. Each value in a bin is replaced by the arithmetic mean of that bin.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

age_data = np.array([
    13, 15, 16, 16, 19, 20, 20, 21, 22,
    22, 25, 25, 25, 25, 30, 33, 33, 35,
    35, 35, 35, 36, 40, 45, 46, 52, 70
], dtype=float)

n = len(age_data)
bin_depth = 3
num_bins = n // bin_depth

# Partition into 9 bins of depth 3
bins = age_data.reshape(num_bins, bin_depth)
bin_means = bins.mean(axis=1)

print(f"{'Bin #':<6} | {'Original Values':<18} | {'Sum':<6} | {'Bin Mean (exact)':<18} | {'Bin Mean (round)':<16}")
print("-" * 72)
for i in range(num_bins):
    b_vals = [int(x) for x in bins[i]]
    b_sum = sum(b_vals)
    b_mean = bin_means[i]
    print(f"Bin {i+1:<2} | {str(b_vals):<18} | {b_sum:<6} | {b_mean:<18.4f} | {b_mean:<16.2f}")

smoothed_means = np.repeat(bin_means, bin_depth)
print(f"\nSmoothed series by bin means:\n{[round(float(x), 2) for x in smoothed_means]}")


In [ ]:
# Plotting Smoothing by Bin Means vs Original Data
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=120)
x = np.arange(1, n + 1)
ax.plot(x, age_data, 'o-', color='#1f77b4', linewidth=2, label='Original Age Data', markersize=6)
ax.step(x, smoothed_means, where='mid', color='#d62728', linewidth=2.5, linestyle='--', label='Smoothed by Bin Means (depth=3)')

for b_idx in range(1, num_bins):
    ax.axvline(b_idx * bin_depth + 0.5, color='gray', linestyle=':', alpha=0.5)

ax.set_title('Problem 2(a): Smoothing by Bin Means (depth=3)', fontsize=12, fontweight='bold')
ax.set_xlabel('Sorted Observation Index')
ax.set_ylabel('Age')
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()


### (b) Outlier Determination in the Data

We evaluate outliers using:
1. **Tukey's IQR Rule (Five-Number Summary):**
   - $Q_1 = 20$, $Q_3 = 35$, $IQR = 15$
   - $\text{Lower Fence} = Q_1 - 1.5 \times IQR = 20 - 22.5 = -2.5$
   - $\text{Upper Fence} = Q_3 + 1.5 \times IQR = 35 + 22.5 = 57.5$
   - Value **70** exceeds $57.5$, confirming it as an outlier.
2. **$3\sigma$ Standard Score Rule:**
   - Sample mean $\bar{x} \approx 29.963$, standard deviation $s \approx 12.942$
   - $z(70) = \frac{70 - 29.963}{12.942} \approx 3.09 > 3.0$, flagging **70** as an outlier.
3. **Modified $Z$-score using Median and MAD:**
   - $\text{Median} = 25$, $\text{MAD} = 9.0$, $M(70) = \frac{0.6745 \times (70 - 25)}{9.0} \approx 3.37$.


In [ ]:
# Outlier detection calculations & visualization
q1 = age_data[6]
q3 = age_data[20]
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

mean_age = np.mean(age_data)
std_age = np.std(age_data, ddof=1)
z_scores = (age_data - mean_age) / std_age

fig, (ax_box, ax_z) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

# Boxplot
ax_box.boxplot(age_data, patch_artist=True,
               boxprops=dict(facecolor='#c6dbef', color='#08519c'),
               medianprops=dict(color='#d94801', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor='red', markersize=8))
ax_box.axhline(upper_fence, color='red', linestyle='--', label=f'Upper Fence = {upper_fence:.1f}')
ax_box.axhline(lower_fence, color='green', linestyle='--', label=f'Lower Fence = {lower_fence:.1f}')
ax_box.text(1.08, 70, 'Outlier: 70', color='red', fontweight='bold', va='center')
ax_box.set_title('Tukey Boxplot Outlier Detection', fontweight='bold')
ax_box.set_ylabel('Age')
ax_box.set_xticks([1])
ax_box.set_xticklabels(['Age'])
ax_box.legend()
ax_box.grid(True, linestyle=':', alpha=0.5)

# Z-score plot
ax_z.stem(x, z_scores, linefmt='b-', markerfmt='bo', basefmt='k-')
ax_z.axhline(3.0, color='red', linestyle='--', label='3-Sigma Threshold (z = 3.0)')
ax_z.axhline(-3.0, color='red', linestyle='--')
ax_z.axhline(2.0, color='orange', linestyle=':', label='2-Sigma Threshold (z = 2.0)')
ax_z.text(27, z_scores[-1], f' z = {z_scores[-1]:.2f}', color='red', fontweight='bold', va='bottom', ha='right')
ax_z.set_title('Z-Score Outlier Analysis', fontweight='bold')
ax_z.set_xlabel('Observation Index')
ax_z.set_ylabel('Z-Score')
ax_z.legend()
ax_z.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


### (c) Alternative Data Smoothing Methods
1. **Smoothing by Bin Medians:** Replace values in each bin with the bin median (e.g., Bin 9 $[46, 52, 70] \to 52$). More robust to intra-bin outliers than bin means.
2. **Smoothing by Bin Boundaries:** Replace values with the closest boundary value (min or max of the bin).
3. **Regression:** Fit parametric mathematical functions ($y = wx + b$ or polynomial models) to capture macro trends and eliminate residual noise.
4. **Clustering:** Partition data points into clusters; points are represented by centroids, and isolated unclustered points are removed as noise.
5. **Moving Average / Exponential Smoothing:** Sliding window or exponentially weighted averaging over sequential series.
6. **Wavelet and Spectral Filtering:** High-frequency noise coefficients in frequency/wavelet space are filtered out.


## Problem P3: Issues in Data Integration

Data integration combines data from multiple heterogeneous sources into a unified repository. The four principal challenges (Han & Kamber, Section 3.3) are:

1. **Schema Integration & Entity Identification:**
   - Determining when two records from separate systems refer to the identical real-world entity (`cust_id` vs `customer_number`).
   - Resolving **synonyms** (different names for the same attribute) and **homonyms** (same name for different attributes).
   - Addressing structural differences across database schemas via metadata repositories.
2. **Redundancy & Correlation Analysis:**
   - Avoiding derived attributes that duplicate information (e.g., storing both `birth_date` and `age`).
   - Detecting correlation using $\chi^2$ test of independence for nominal attributes, and Pearson correlation coefficient ($r$) or covariance for numeric attributes.
3. **Tuple Duplication:**
   - Identifying and resolving duplicate or near-duplicate records using record linkage and string similarity metrics (Levenshtein, Jaro-Winkler).
4. **Data Value Conflicts & Semantic Inconsistency:**
   - Differences in measurement units (kg vs lb), currency (USD vs VND), scale, or representation encodings.
   - Temporal divergence caused by varying update frequencies across systems.


## Problem P4: Value Ranges of Normalization Methods

| Normalization Method | Formula | Value Range | Notes |
|:---|:---:|:---:|:---|
| **Min-Max Normalization** | $v' = \frac{v - \min_A}{\max_A - \min_A}(\text{new\_max}_A - \text{new\_min}_A) + \text{new\_min}_A$ | $[\text{new\_min}_A, \text{new\_max}_A]$ | Strictly $[0.0, 1.0]$ when mapped onto $[0, 1]$. Future unseen points may cause out-of-bounds errors. |
| **Z-Score Normalization** | $v' = \frac{v - \bar{A}}{\sigma_A}$ | $(-\infty, +\infty)$ | The entire real line $\mathbb{R}$. Empirically $\approx 99.73\%$ within $[-3, +3]$, but mathematically unbounded. |
| **Z-Score with MAD** | $v' = \frac{v - \bar{A}}{s_A}$ where $s_A = \frac{1}{n}\sum |v_i - \bar{A}|$ | $(-\infty, +\infty)$ | The entire real line $\mathbb{R}$. Unbounded; more resistant to outliers because deviations are not squared. |
| **Decimal Scaling** | $v' = \frac{v}{10^j}$ where $j = \min\{k \in \mathbb{Z} : \max(|v|) < 10^k\}$ | $(-1.0, 1.0)$ | Strictly bounded within open interval $(-1, 1)$. For non-negative values, $[0.0, 1.0)$. |


## Problem P5: Normalization Calculations on Age Data ($v = 35$)

Dataset properties:
- $\min = 13$, $\max = 70$
- Mean: $\bar{x}_{\text{exact}} = \frac{809}{27} \approx 29.9630$ (rounded: $29.96$)
- Standard deviation: $s = 12.94$ years
- Target value: $v = 35$


In [ ]:
# Normalization calculations
v = 35.0
min_age = 13.0
max_age = 70.0
mean_age_round = 29.96
mean_age_exact = 809.0 / 27.0
std_age = 12.94

# (a) Min-Max onto [0.0, 1.0]
norm_minmax_35 = (v - min_age) / (max_age - min_age)

# (b) Z-Score
norm_zscore_round = (v - mean_age_round) / std_age
norm_zscore_exact = (v - mean_age_exact) / std_age

# (c) Decimal scaling: max = 70 -> j = 2 (10^2 = 100)
j = 2
norm_decimal_35 = v / (10**j)

print(f"(a) Min-Max Normalization:      v' = (35 - 13) / (70 - 13) = 22 / 57 = {norm_minmax_35:.6f} ≈ {norm_minmax_35:.4f}")
print(f"(b) Z-Score (rounded mean):     v' = (35 - 29.96) / 12.94  = 5.04 / 12.94 = {norm_zscore_round:.6f} ≈ {norm_zscore_round:.4f}")
print(f"    Z-Score (exact mean):       v' = (35 - 29.9630) / 12.94 = {norm_zscore_exact:.6f} ≈ {norm_zscore_exact:.4f}")
print(f"(c) Decimal Scaling (j=2):      v' = 35 / 100 = {norm_decimal_35:.4f}")


In [ ]:
# Comparison visualization across normalization methods
norm_all_minmax = (age_data - min_age) / (max_age - min_age)
norm_all_zscore = (age_data - mean_age_exact) / std_age
norm_all_decimal = age_data / (10**j)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=120)

axes[0, 0].hist(age_data, bins=10, color='#1f77b4', edgecolor='black', alpha=0.7, rwidth=0.85)
axes[0, 0].axvline(35, color='red', linestyle='--', linewidth=2, label='v = 35')
axes[0, 0].set_title('Original Age Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle=':', alpha=0.5)

axes[0, 1].hist(norm_all_minmax, bins=10, color='#ff7f0e', edgecolor='black', alpha=0.7, rwidth=0.85)
axes[0, 1].axvline(norm_minmax_35, color='red', linestyle='--', linewidth=2, label=f"v' = {norm_minmax_35:.4f}")
axes[0, 1].set_title('Min-Max Normalized [0.0, 1.0]', fontweight='bold')
axes[0, 1].set_xlabel('Normalized Value')
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle=':', alpha=0.5)

axes[1, 0].hist(norm_all_zscore, bins=10, color='#2ca02c', edgecolor='black', alpha=0.7, rwidth=0.85)
axes[1, 0].axvline(norm_zscore_exact, color='red', linestyle='--', linewidth=2, label=f"v' = {norm_zscore_exact:.4f}")
axes[1, 0].set_title('Z-Score Normalized', fontweight='bold')
axes[1, 0].set_xlabel('Z-Score')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle=':', alpha=0.5)

axes[1, 1].hist(norm_all_decimal, bins=10, color='#9467bd', edgecolor='black', alpha=0.7, rwidth=0.85)
axes[1, 1].axvline(norm_decimal_35, color='red', linestyle='--', linewidth=2, label=f"v' = {norm_decimal_35:.4f}")
axes[1, 1].set_title('Decimal Scaling Normalized', fontweight='bold')
axes[1, 1].set_xlabel('Normalized Value')
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


### (d) Method Preference & Justification

**Preferred Method:** **Z-Score Normalization** (or Z-score using MAD).

**Reasons:**
1. **Resistance to Outlier Distortion:** The presence of outlier **70** inflates the Min-Max range ($70 - 13 = 57$). Consequently, 26 out of 27 values ($96.3\%$) are compressed into the narrow interval $[0.0, 0.684]$, squashing normal data variance. Decimal scaling is likewise dictated by the power of 10 required to cover $70$.
2. **No Out-of-Bounds Error:** Future observations outside $[13, 70]$ produce out-of-bounds values under Min-Max scaling. Z-score normalization handles arbitrary new observations without boundary violations.
3. **Statistical Alignment:** Z-score normalization centers data at zero and scales to unit variance, which is required for machine learning algorithms (PCA, k-means, SVM, neural networks).


## Problem P6: Automatic Concept Hierarchy Generation for Nominal Data

### Algorithmic Principle
Higher-level concepts encompass broader abstractions and thus contain fewer distinct values than subordinate lower-level concepts (Han & Kamber, Section 3.5.6).
- **Heuristic Rule:** Sort nominal attributes in ascending order of their distinct value counts:
  $$C_{(1)} \le C_{(2)} \le \dots \le C_{(m)}$$
- The attribute with the lowest count forms Level 1 (Top / Root), while the attribute with the highest count forms the Leaf (Bottom).


In [ ]:
from typing import Dict, List, Tuple

class ConceptHierarchyGenerator:
    """
    Automatic concept hierarchy generator for nominal schema attributes
    based on the number of distinct values per attribute (Han & Kamber, Sec 3.5.6).
    """
    def __init__(self, schema_name: str):
        self.schema_name = schema_name
        self.attributes: Dict[str, int] = {}
        
    def add_attribute(self, attr_name: str, distinct_count: int):
        if distinct_count < 1:
            raise ValueError(f"Distinct count must be >= 1 for {attr_name}")
        self.attributes[attr_name] = distinct_count
        
    def generate_hierarchy(self) -> List[Tuple[str, int]]:
        return sorted(self.attributes.items(), key=lambda item: item[1])
        
    def display_hierarchy(self):
        ordered = self.generate_hierarchy()
        print(f"Concept Hierarchy for: '{self.schema_name}'")
        print("=" * 60)
        m = len(ordered)
        for level, (attr, count) in enumerate(ordered):
            pos = '(Top - General)' if level == 0 else ('(Bottom - Detailed)' if level == m - 1 else '(Intermediate)')
            indent = '    ' * level
            arrow = '└── ' if level > 0 else '┌── '
            print(f"{indent}{arrow}Level {level + 1}: {attr:<20} [{count:>10,d} distinct values]  {pos}")
        print("=" * 60)
        chain = ' < '.join([attr for attr, _ in reversed(ordered)])
        print(f"Hierarchical Ordering (Specific < General):\n  {chain}\n")

# Example 1: Location Dimension (Textbook Example 3.7)
loc = ConceptHierarchyGenerator("Location Dimension")
loc.add_attribute("country", 15)
loc.add_attribute("province_or_state", 365)
loc.add_attribute("city", 3567)
loc.add_attribute("street", 674339)
loc.display_hierarchy()

# Example 2: Product Taxonomy Dimension
prod = ConceptHierarchyGenerator("Product Dimension")
prod.add_attribute("product_line", 8)
prod.add_attribute("category", 42)
prod.add_attribute("brand", 250)
prod.add_attribute("item_sku", 18500)
prod.display_hierarchy()
